# SpoilCheck v2 — Multimodal Spoilage Training (fixed TR-6 dataset)

Run the cells **top to bottom** on a **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU).

| Cell | What it does | Approx. time |
|---|---|---|
| 1 | Mount Drive, unzip the fixed dataset | ~5–10 min |
| 2 | Pair sRGB ↔ IR ↔ gas, build ONE shared split (all 12 classes in val + test) | <1 min |
| 3 | Cache small sRGB copies, fit scalers, shared helpers | ~5 min |
| 4 | Model 1 — sRGB ConvNeXt-Tiny | ~15–25 min |
| 5 | Model 2 — IR ResNet18 + temperature MLP | ~10 min |
| 6 | Model 3 — Gas MLP + honest baselines | <1 min |
| 7 | Model 4 — Late-fusion head (fixed modality dropout) + sensor-unplugged test | ~3 min |
| 8 | Ablation table (sRGB / +IR / +Gas / all, 3 seeds) | ~3 min |
| 9 | Grad-CAM: what the sRGB model looks at | <1 min |
| 10 | Final summary + backend inference helper | <1 min |

All outputs go to a **new** Drive folder `MyDrive/spoilcheck_v2/`, so your old models are not overwritten.

## Cell 1 — Setup: mount Drive, unzip the fixed dataset
Upload `SpoilCheck_TR6_fixed.zip` to the root of **My Drive** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install timm

import os, random
import numpy as np
import torch

ZIP_ON_DRIVE = "/content/drive/MyDrive/SpoilCheck_TR6_fixed.zip"   # the fixed dataset zip
DATA_DIR     = "/content/dataset"                                   # unzipped here (fast local disk)
CACHE_DIR    = "/content/cache"                                     # small sRGB copies for fast training
SAVE_DIR     = "/content/drive/MyDrive/spoilcheck_v2"               # NEW output folder (old models kept)
os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    assert os.path.exists(ZIP_ON_DRIVE), f"Upload the zip to Drive first: {ZIP_ON_DRIVE}"
    !cp "{ZIP_ON_DRIVE}" /content/data.zip
    !unzip -q /content/data.zip -d "{DATA_DIR}"
    !rm /content/data.zip
print("Varieties:", sorted(os.listdir(DATA_DIR)))

SEED = 42
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, torch.cuda.get_device_name(0) if USE_AMP else "(no GPU! switch runtime to T4)")

## Cell 2 — Build the paired multimodal table + ONE shared split
* **sRGB ↔ IR:** nearest IR frame in time from the same class folder; photos with **no IR frame within 30 min are dropped** (the old code paired frames up to 3.7 h apart).
* **Gas:** the log has no timestamps, only "Day" blocks. Blocks are read **in order of appearance** and matched to the n-th photo date; one reading of that day is sampled. This is **day-level context**, not a per-photo measurement.
* **Split:** StratifiedGroupKFold by session (`Variety_State_Date_Hour`) so burst/angle shots never cross train/test. The first seed whose val **and** test contain **all 12 classes** is used (chosen on class coverage only, never on accuracy).
* All 4 models use this **same** split.

In [ ]:
import re, glob, json, bisect
import pandas as pd
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold

MAX_GAP_MIN = 30        # max minutes between an sRGB photo and its IR frame
GROUP_LEVEL = "hour"    # "hour" = Variety_State_Date_Hour (default) | "date" = stricter: whole days held out

def parse_gas_blocks(path):
    blocks = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if re.match(r"^Day\s+\d+", line, re.I):
                blocks.append([])
                continue
            m = re.search(r"ADC:\s*([\d.]+).*?Rs:\s*([\d.]+).*?Rs/R0:\s*([\d.]+).*?Methane:\s*([\d.]+)", line, re.I)
            if m:
                if not blocks:
                    blocks.append([])
                blocks[-1].append(tuple(float(x) for x in m.groups()))
    return [b for b in blocks if b]

rng = np.random.default_rng(SEED)
rows, dropped = [], {}
for variety in sorted(os.listdir(DATA_DIR)):
    for state in ["Not_spoiled", "Spoiled"]:
        base = os.path.join(DATA_DIR, variety, state)
        if not os.path.isdir(base):
            continue
        label_name = f"{variety}_{state}"

        ir = []
        for p in glob.glob(os.path.join(base, "IR_fusion_images", "*.jpg")):
            m = re.match(r"(\d{8}_\d{6})_([\d.]+)C_([\d.]+)C", os.path.basename(p))
            if m:
                t = datetime.strptime(m.group(1), "%Y%m%d_%H%M%S").timestamp()
                ir.append((t, p, float(m.group(2)), float(m.group(3))))
        ir.sort()
        ir_ts = [r[0] for r in ir]

        gas_blocks = parse_gas_blocks(os.path.join(base, "Methane_gas_readings.txt"))
        srgb = sorted(glob.glob(os.path.join(base, "sRGB_images", "*.jpg")))
        dates = sorted({os.path.basename(p)[:8] for p in srgb})
        date_to_block = {d: min(i, len(gas_blocks) - 1) for i, d in enumerate(dates)}

        n_drop = 0
        for p in srgb:
            m = re.match(r"(\d{8}_\d{6})", os.path.basename(p))
            if not m or not ir:
                n_drop += 1
                continue
            dt = datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
            t = dt.timestamp()
            i = bisect.bisect_left(ir_ts, t)
            j = min([k for k in (i - 1, i) if 0 <= k < len(ir)], key=lambda k: abs(ir_ts[k] - t))
            gap_min = abs(ir_ts[j] - t) / 60
            if gap_min > MAX_GAP_MIN:
                n_drop += 1
                continue
            _, ir_path, t_min, t_max = ir[j]
            block = gas_blocks[date_to_block[dt.strftime("%Y%m%d")]]
            adc, rs, rs_r0, ppm = block[rng.integers(len(block))]
            rows.append(dict(
                srgb_path=p, ir_path=ir_path, label_name=label_name,
                binary_label=int(state == "Spoiled"),
                date=dt.strftime("%Y%m%d"), hour=dt.strftime("%H"), ir_gap_min=round(gap_min, 1),
                t_min=t_min, t_max=t_max, t_diff=round(t_max - t_min, 2),
                adc=adc, rs=rs, rs_r0=rs_r0, methane_ppm=ppm))
        dropped[label_name] = n_drop

df = pd.DataFrame(rows)
classes = sorted(df["label_name"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)
df["label"] = df["label_name"].map(class_to_idx)
df["group"] = df["label_name"] + "_" + df["date"] + (("_" + df["hour"]) if GROUP_LEVEL == "hour" else "")
print(f"Paired samples kept: {len(df)} | dropped (no IR within {MAX_GAP_MIN} min):",
      {k: v for k, v in dropped.items() if v})

def make_split(seed):
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, tmp_idx = next(outer.split(df, df["label"], groups=df["group"]))
    tmp = df.iloc[tmp_idx]
    inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)
    va_idx, te_idx = next(inner.split(tmp, tmp["label"], groups=tmp["group"]))
    return df.iloc[tr_idx], tmp.iloc[va_idx], tmp.iloc[te_idx]

for split_seed in range(500):
    tr, va, te = make_split(split_seed)
    if va["label"].nunique() == NUM_CLASSES and te["label"].nunique() == NUM_CLASSES:
        break
else:
    print("WARNING: no split found with all classes in val AND test (expected with GROUP_LEVEL='date').")
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (tr, va, te))
assert not set(train_df.group) & set(test_df.group), "group leakage train/test"
assert not set(train_df.group) & set(val_df.group), "group leakage train/val"

print(f"Split seed {split_seed} | Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")
print(pd.DataFrame({"train": train_df.label_name.value_counts(), "val": val_df.label_name.value_counts(),
                    "test": test_df.label_name.value_counts()}).fillna(0).astype(int))

with open(os.path.join(SAVE_DIR, "class_indices.json"), "w") as f:
    json.dump(class_to_idx, f, indent=2)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(os.path.join(SAVE_DIR, f"mm_{name}.csv"), index=False)

## Cell 3 — Cache small sRGB copies, fit scalers, shared helpers
The phone photos are 4000×3000; decoding them every epoch is what made the old training slow. They are resized **once** (short side 256) — the model still sees the same 224×224 input as before.

In [ ]:
import shutil, joblib
import torch.nn as nn
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

# ---- 1. one-time sRGB cache ----
os.makedirs(CACHE_DIR, exist_ok=True)
def cache_path(p):
    return os.path.join(CACHE_DIR, os.path.relpath(p, DATA_DIR).replace("/", "__"))
def make_small(p, short=256):
    out = cache_path(p)
    if os.path.exists(out):
        return
    with Image.open(p) as im:
        im.draft("RGB", (im.width // 4, im.height // 4))     # fast reduced-size JPEG decode
        im = im.convert("RGB")
        s = short / min(im.size)
        im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC).save(out, quality=95)
all_srgb = pd.concat([train_df, val_df, test_df])["srgb_path"].unique()
with ThreadPoolExecutor(8) as ex:
    list(tqdm(ex.map(make_small, all_srgb), total=len(all_srgb), desc="Caching sRGB"))
for d in (train_df, val_df, test_df):
    d["srgb_small"] = d["srgb_path"].map(cache_path)

# ---- 2. sensor scalers (fit on TRAIN only) ----
temp_cols = ["t_min", "t_max", "t_diff"]
gas_cols  = ["adc", "rs", "rs_r0", "methane_ppm"]
temp_scaler = StandardScaler().fit(train_df[temp_cols].values)
gas_scaler  = StandardScaler().fit(train_df[gas_cols].values)
joblib.dump({"temp_scaler": temp_scaler, "gas_scaler": gas_scaler}, os.path.join(SAVE_DIR, "sensor_scalers.pkl"))

# ---- 3. transforms (colour-safe: no hue/saturation/brightness jitter) ----
IMG_SIZE, BATCH_SIZE, NUM_WORKERS = 224, 64, 2     # set NUM_WORKERS = 0 if the DataLoader hangs
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
srgb_train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
ir_train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# ---- 4. dataset: (srgb, ir, temps, gas, label); a modality with transform=None is skipped ----
class PairedDataset(Dataset):
    def __init__(self, df, srgb_tfms=None, ir_tfms=None):
        self.df = df.reset_index(drop=True)
        self.srgb_tfms, self.ir_tfms = srgb_tfms, ir_tfms
        self.temps = torch.tensor(temp_scaler.transform(self.df[temp_cols].values), dtype=torch.float32)
        self.gas   = torch.tensor(gas_scaler.transform(self.df[gas_cols].values), dtype=torch.float32)
        self.y     = torch.tensor(self.df["label"].values, dtype=torch.long)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        srgb = self.srgb_tfms(Image.open(self.df.loc[i, "srgb_small"]).convert("RGB")) if self.srgb_tfms else torch.zeros(0)
        ir   = self.ir_tfms(Image.open(self.df.loc[i, "ir_path"]).convert("RGB")) if self.ir_tfms else torch.zeros(0)
        return srgb, ir, self.temps[i], self.gas[i], self.y[i]

def make_loader(df, srgb_tfms=None, ir_tfms=None, train=False):
    return DataLoader(PairedDataset(df, srgb_tfms, ir_tfms), batch_size=BATCH_SIZE, shuffle=train,
                      drop_last=train, num_workers=NUM_WORKERS, pin_memory=True)

# ---- 5. class weights, metrics helpers ----
counts = np.bincount(train_df["label"], minlength=NUM_CLASSES)
class_weights = torch.tensor(1.0 / np.maximum(counts, 1), dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * NUM_CLASSES).to(device)
present_test = sorted(test_df["label"].unique())
is_spoiled = np.array([c.endswith("_Spoiled") for c in classes])
RESULTS = []

@torch.no_grad()
def predict_logits(model, forward_fn, dl):
    model.eval()
    L, Y = [], []
    for batch in dl:
        batch = [b.to(device, non_blocking=True) for b in batch]
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            out = forward_fn(model, batch)
        L.append(out.float().cpu()); Y.append(batch[-1].cpu())
    return torch.cat(L), torch.cat(Y)

def run_training(model, forward_fn, train_dl, val_dl, epochs, lr, save_path, name):
    # AdamW + cosine LR + AMP; the best epoch is chosen by VALIDATION LOSS
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, hist = float("inf"), []
    for ep in range(epochs):
        model.train()
        tl, tc, n = 0.0, 0, 0
        for batch in tqdm(train_dl, desc=f"{name} {ep + 1}/{epochs}", leave=False):
            batch = [b.to(device, non_blocking=True) for b in batch]
            y = batch[-1]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                out = forward_fn(model, batch)
                loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); n += len(y)
        sched.step()
        vlogits, vy = predict_logits(model, forward_fn, val_dl)
        vloss = F.cross_entropy(vlogits, vy).item()
        vacc = (vlogits.argmax(1) == vy).float().mean().item()
        vf1 = f1_score(vy, vlogits.argmax(1), average="macro", zero_division=0)
        hist.append(dict(epoch=ep + 1, train_loss=tl / n, train_acc=tc / n, val_loss=vloss, val_acc=vacc, val_f1=vf1))
        flag = ""
        if vloss < best:
            best = vloss
            torch.save(model.state_dict(), save_path)
            flag = "  <-- best (saved)"
        print(f"{name} [{ep + 1:2d}/{epochs}] train loss {tl / n:.4f} acc {tc / n:.4f} | "
              f"val loss {vloss:.4f} acc {vacc:.4f} macro-F1 {vf1:.4f}{flag}")
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    return model, pd.DataFrame(hist)

def report(name, preds, y, png=None):
    preds, y = np.asarray(preds), np.asarray(y)
    acc = (preds == y).mean() * 100
    mf1 = f1_score(y, preds, labels=present_test, average="macro", zero_division=0) * 100
    bin_acc = (is_spoiled[preds] == is_spoiled[y]).mean() * 100
    print(f"\n=== {name} | Test acc {acc:.2f}% | macro-F1 {mf1:.2f}% | spoiled-vs-fresh {bin_acc:.2f}% ===")
    print(classification_report(y, preds, labels=present_test,
                                target_names=[classes[i] for i in present_test], zero_division=0))
    if png:
        fig, ax = plt.subplots(figsize=(11, 9))
        ConfusionMatrixDisplay(confusion_matrix(y, preds, labels=present_test),
                               display_labels=[classes[i] for i in present_test]).plot(
            ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
        ax.set_title(f"{name} (acc {acc:.2f}%)")
        plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, png), dpi=200); plt.show()
    return dict(model=name, acc=round(acc, 2), macro_f1=round(mf1, 2), spoiled_vs_fresh=round(bin_acc, 2))

print("Ready.", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})

## Cell 4 — Model 1: sRGB ConvNeXt-Tiny

In [ ]:
seed_everything()
srgb_model = timm.create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES, drop_rate=0.2).to(device)
fwd_srgb = lambda m, b: m(b[0])

srgb_model, hist_srgb = run_training(
    srgb_model, fwd_srgb,
    make_loader(train_df, srgb_tfms=srgb_train_tfms, train=True),
    make_loader(val_df, srgb_tfms=eval_tfms),
    epochs=15, lr=2e-4, save_path=os.path.join(SAVE_DIR, "best_convnext_spoilage.pth"), name="sRGB")

logits, y = predict_logits(srgb_model, fwd_srgb, make_loader(test_df, srgb_tfms=eval_tfms))
RESULTS.append(report("Model 1: sRGB ConvNeXt-Tiny", logits.argmax(1).numpy(), y.numpy(), "cm_srgb.png"))

## Cell 5 — Model 2: IR ResNet18 + temperature MLP

In [ ]:
class StandaloneIRModel(nn.Module):
    def __init__(self, num_classes=12, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)   # 512-d
        self.temp_net = nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(512 + 32, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, ir_img, temps, return_features=False):
        fused = torch.cat([self.backbone(ir_img), self.temp_net(temps)], dim=1)    # 544-d
        return fused if return_features else self.head(fused)

seed_everything()
ir_model = StandaloneIRModel(NUM_CLASSES).to(device)
fwd_ir = lambda m, b: m(b[1], b[2])

ir_model, hist_ir = run_training(
    ir_model, fwd_ir,
    make_loader(train_df, ir_tfms=ir_train_tfms, train=True),
    make_loader(val_df, ir_tfms=eval_tfms),
    epochs=12, lr=3e-4, save_path=os.path.join(SAVE_DIR, "best_ir_model.pth"), name="IR")

logits, y = predict_logits(ir_model, fwd_ir, make_loader(test_df, ir_tfms=eval_tfms))
RESULTS.append(report("Model 2: IR ResNet18 + temps", logits.argmax(1).numpy(), y.numpy(), "cm_ir.png"))

## Cell 6 — Model 3: Gas MLP + honest baselines
Mini-batch training (the old version used one full batch per epoch and saved after 2 steps).

In [ ]:
class StandaloneGasMLP(nn.Module):
    def __init__(self, in_dim=4, out_dim=12):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.head = nn.Linear(64, out_dim)
    def forward(self, x, return_features=False):
        feat = self.net(x)                                                          # 64-d
        return feat if return_features else self.head(feat)

def gas_loader(df, train=False):
    X = torch.tensor(gas_scaler.transform(df[gas_cols].values), dtype=torch.float32)
    y = torch.tensor(df["label"].values, dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=256, shuffle=train, drop_last=train)

seed_everything()
gas_model = StandaloneGasMLP(4, NUM_CLASSES).to(device)
fwd_gas = lambda m, b: m(b[0])
gas_model, hist_gas = run_training(
    gas_model, fwd_gas, gas_loader(train_df, train=True), gas_loader(val_df),
    epochs=100, lr=1e-3, save_path=os.path.join(SAVE_DIR, "best_gas_model.pth"), name="Gas")

logits, y = predict_logits(gas_model, fwd_gas, gas_loader(test_df))
RESULTS.append(report("Model 3: Gas MLP", logits.argmax(1).numpy(), y.numpy()))

y_np = test_df["label"].values
print(f"Baseline - always predict the most common class : {np.bincount(y_np).max() / len(y_np) * 100:.2f}%")
print(f"Baseline - always predict 'fresh'               : {(~is_spoiled[y_np]).mean() * 100:.2f}%   "
      "<- compare with the Gas MLP 'spoiled-vs-fresh' number above")

## Cell 7 — Model 4: Late-fusion head
* Encoders are frozen; their embeddings are cached once (sRGB 768 + IR 544 + Gas 64).
* Each branch is projected (256 / 128 / 64) → concatenated (448) → classifier.
* **Fixed:** a missing sensor zeros the *projected* features (`use_ir=False` / `use_gas=False`) — the same tensor modality dropout zeros during training. Use these flags in the backend too.
* Modality dropout is per-sample; best epoch is chosen by validation loss.

In [ ]:
# ---- 1. frozen encoders -> cached embeddings ----
srgb_enc = timm.create_model("convnext_tiny", pretrained=False, num_classes=NUM_CLASSES)
srgb_enc.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_convnext_spoilage.pth"), map_location="cpu"))
srgb_enc.reset_classifier(0)
srgb_enc = srgb_enc.to(device).eval()
ir_enc = ir_model.eval()
gas_enc = gas_model.eval()

@torch.no_grad()
def extract(df, desc):
    out = {"srgb": [], "ir": [], "gas": [], "y": []}
    for srgb, ir, temps, gas, y in tqdm(make_loader(df, srgb_tfms=eval_tfms, ir_tfms=eval_tfms), desc=desc, leave=False):
        srgb, ir, temps, gas = srgb.to(device), ir.to(device), temps.to(device), gas.to(device)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            out["srgb"].append(srgb_enc(srgb).float())
            out["ir"].append(ir_enc(ir, temps, return_features=True).float())
            out["gas"].append(gas_enc(gas, return_features=True).float())
        out["y"].append(y.to(device))
    return {k: torch.cat(v) for k, v in out.items()}

EMB = {"train": extract(train_df, "Train emb"), "val": extract(val_df, "Val emb"), "test": extract(test_df, "Test emb")}
torch.save({s: {k: v.cpu() for k, v in d.items()} for s, d in EMB.items()}, os.path.join(SAVE_DIR, "fusion_embeddings_cache.pt"))
print({s: tuple(EMB[s]["srgb"].shape) for s in EMB}, "| IR", EMB["train"]["ir"].shape[1], "| Gas", EMB["train"]["gas"].shape[1])

# ---- 2. fusion network ----
class MultimodalFusionNet(nn.Module):
    def __init__(self, num_classes=12, p_mod_drop=0.0):
        super().__init__()
        self.proj_srgb = nn.Sequential(nn.Linear(768, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3))
        self.proj_ir   = nn.Sequential(nn.Linear(544, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.3))
        self.proj_gas  = nn.Sequential(nn.Linear(64, 64),   nn.LayerNorm(64),  nn.ReLU(), nn.Dropout(0.2))
        self.fusion_head = nn.Sequential(nn.Linear(256 + 128 + 64, 192), nn.BatchNorm1d(192), nn.ReLU(),
                                         nn.Dropout(0.3), nn.Linear(192, num_classes))
        self.p_mod_drop = p_mod_drop
    def forward(self, e_srgb, e_ir, e_gas, use_ir=True, use_gas=True):
        h_srgb, h_ir, h_gas = self.proj_srgb(e_srgb), self.proj_ir(e_ir), self.proj_gas(e_gas)
        if not use_ir:
            h_ir = torch.zeros_like(h_ir)
        if not use_gas:
            h_gas = torch.zeros_like(h_gas)
        if self.training and self.p_mod_drop > 0:          # per-sample modality dropout
            B = h_srgb.size(0)
            h_ir  = h_ir  * (torch.rand(B, 1, device=h_ir.device)  >= self.p_mod_drop).float()
            h_gas = h_gas * (torch.rand(B, 1, device=h_gas.device) >= self.p_mod_drop).float()
        return self.fusion_head(torch.cat([h_srgb, h_ir, h_gas], dim=1))

fusion_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

def train_fusion(use_ir=True, use_gas=True, p_mod_drop=0.15, seed=0, epochs=60, bs=128):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va = EMB["train"], EMB["val"]
    model = MultimodalFusionNet(NUM_CLASSES, p_mod_drop).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    n = len(tr["y"]); best, best_state, best_ep = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n - 1, bs):
            idx = perm[i:i + bs]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss = fusion_criterion(model(tr["srgb"][idx], tr["ir"][idx], tr["gas"][idx], use_ir, use_gas), tr["y"][idx])
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vloss = F.cross_entropy(model(va["srgb"], va["ir"], va["gas"], use_ir, use_gas), va["y"]).item()
        if vloss < best:
            best, best_ep = vloss, ep + 1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    return model, best_ep

@torch.no_grad()
def fusion_preds(model, use_ir=True, use_gas=True):
    te = EMB["test"]
    return model(te["srgb"], te["ir"], te["gas"], use_ir, use_gas).argmax(1).cpu().numpy()

# ---- 3. train the final fusion model ----
fusion_model, best_ep = train_fusion(use_ir=True, use_gas=True, p_mod_drop=0.15, seed=0)
torch.save(fusion_model.state_dict(), os.path.join(SAVE_DIR, "best_multimodal_fusion.pth"))
print(f"Fusion best epoch: {best_ep}/60")

y_te = EMB["test"]["y"].cpu().numpy()
RESULTS.append(report("Model 4: Fusion (sRGB+IR+Gas)", fusion_preds(fusion_model), y_te, "cm_fusion.png"))
for tag, ui, ug in [("IR unplugged", False, True), ("Gas unplugged", True, False), ("IR+Gas unplugged", False, False)]:
    p = fusion_preds(fusion_model, ui, ug)
    print(f"Fusion with {tag:17s}: acc {(p == y_te).mean() * 100:.2f}% | "
          f"macro-F1 {f1_score(y_te, p, labels=present_test, average='macro', zero_division=0) * 100:.2f}%")

## Cell 8 — Ablation table (the evidence for "why multimodal")
Same fusion head, same training — only the allowed sensors change. 3 seeds → mean ± std.

In [ ]:
CONFIGS = [  # name, use_ir, use_gas, modality dropout
    ("A. sRGB only",                        False, False, 0.00),
    ("B. sRGB + IR",                        True,  False, 0.00),
    ("C. sRGB + Gas",                       False, True,  0.00),
    ("D. sRGB + IR + Gas",                  True,  True,  0.00),
    ("E. sRGB + IR + Gas (modality drop)",  True,  True,  0.15),
]
UNPLUG = [("IR unplugged", False, True), ("Gas unplugged", True, False), ("IR+Gas unplugged", False, False)]
rows = []
for name, ui, ug, p in CONFIGS:
    for seed in [0, 1, 2]:
        m, ep = train_fusion(ui, ug, p, seed)
        tests = [(name, ui, ug)] + ([(f"{name[:2]} -> {t}", a, b) for t, a, b in UNPLUG] if ui and ug else [])
        for label, a, b in tests:
            pr = fusion_preds(m, a, b)
            rows.append(dict(config=label, seed=seed, best_epoch=ep,
                             acc=(pr == y_te).mean() * 100,
                             macro_f1=f1_score(y_te, pr, labels=present_test, average="macro", zero_division=0) * 100,
                             spoiled_vs_fresh=(is_spoiled[pr] == is_spoiled[y_te]).mean() * 100,
                             n_wrong=int((pr != y_te).sum())))
    print("done:", name)

abl = pd.DataFrame(rows)
abl.to_csv(os.path.join(SAVE_DIR, "fusion_ablation_results.csv"), index=False)
order = list(dict.fromkeys(abl["config"]))
metrics = ["acc", "macro_f1", "spoiled_vs_fresh", "n_wrong", "best_epoch"]
g = abl.groupby("config", sort=False)[metrics].agg(["mean", "std"]).loc[order]
table = pd.DataFrame({m: g[(m, "mean")].map("{:.2f}".format) + " ± " + g[(m, "std")].map("{:.2f}".format) for m in metrics})
print(f"\nABLATION on {len(y_te)} test samples, mean ± std over 3 seeds")
print(table.to_string())

## Cell 9 — Grad-CAM: where does the sRGB model look?
Good slide material: heat should sit on spots / mould / colour change, not on the background.

In [ ]:
srgb_model.eval()
_store = {}
def _fwd_hook(module, inp, out):
    _store["act"] = out
    out.register_hook(lambda g: _store.__setitem__("grad", g))
hook = srgb_model.stages[-1].register_forward_hook(_fwd_hook)

def gradcam(x):
    srgb_model.zero_grad()
    with torch.enable_grad():
        out = srgb_model(x.unsqueeze(0).to(device))
        cls = out.argmax(1).item()
        out[0, cls].backward()
    A, G = _store["act"][0], _store["grad"][0]
    cam = F.relu((G.mean(dim=(1, 2), keepdim=True) * A).sum(0))
    cam = F.interpolate(cam[None, None], size=(IMG_SIZE, IMG_SIZE), mode="bilinear")[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), cls

samples = pd.concat([test_df[test_df.binary_label == 0].sample(4, random_state=1),
                     test_df[test_df.binary_label == 1].sample(4, random_state=1)])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, r) in zip(axes.flat, samples.iterrows()):
    img = Image.open(r["srgb_small"]).convert("RGB")
    cam, cls = gradcam(eval_tfms(img))
    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE))); ax.imshow(cam, cmap="jet", alpha=0.4); ax.axis("off")
    ok = "OK" if classes[cls] == r["label_name"] else "WRONG"
    ax.set_title(f"true: {r['label_name']}\npred: {classes[cls]} ({ok})", fontsize=9)
hook.remove()
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "gradcam_examples.png"), dpi=200); plt.show()

## Cell 10 — Final summary + backend inference helper
Copy `SpoilCheckPredictor` (plus the 3 model classes above) into the backend. When a sensor is missing, call `predict(...)` without it — the flags are handled for you.

In [ ]:
summary = pd.DataFrame(RESULTS)
summary.to_csv(os.path.join(SAVE_DIR, "test_results_summary.csv"), index=False)
print(summary.to_string(index=False))

class SpoilCheckPredictor:
    def __init__(self, model_dir, device="cpu"):
        self.device = torch.device(device)
        with open(os.path.join(model_dir, "class_indices.json")) as f:
            c2i = json.load(f)
        self.classes = [k for k, _ in sorted(c2i.items(), key=lambda x: x[1])]
        n = len(self.classes)
        sc = joblib.load(os.path.join(model_dir, "sensor_scalers.pkl"))
        self.temp_scaler, self.gas_scaler = sc["temp_scaler"], sc["gas_scaler"]
        load = lambda m, f: m.load_state_dict(torch.load(os.path.join(model_dir, f), map_location="cpu"))
        self.srgb = timm.create_model("convnext_tiny", pretrained=False, num_classes=n)
        load(self.srgb, "best_convnext_spoilage.pth"); self.srgb.reset_classifier(0)
        self.ir = StandaloneIRModel(n, pretrained=False); load(self.ir, "best_ir_model.pth")
        self.gas = StandaloneGasMLP(4, n); load(self.gas, "best_gas_model.pth")
        self.fusion = MultimodalFusionNet(n); load(self.fusion, "best_multimodal_fusion.pth")
        for m in (self.srgb, self.ir, self.gas, self.fusion):
            m.to(self.device).eval()
        self.tfms = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                                        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    @torch.no_grad()
    def predict(self, srgb_img, ir_img=None, t_min=None, t_max=None, gas=None):
        # srgb_img / ir_img: PIL images. gas: dict with adc, rs, rs_r0, methane_ppm. Missing -> sensor treated as unplugged.
        d = self.device
        e_srgb = self.srgb(self.tfms(srgb_img.convert("RGB")).unsqueeze(0).to(d))
        use_ir = ir_img is not None and t_min is not None and t_max is not None
        if use_ir:
            temps = torch.tensor(self.temp_scaler.transform([[t_min, t_max, t_max - t_min]]), dtype=torch.float32, device=d)
            e_ir = self.ir(self.tfms(ir_img.convert("RGB")).unsqueeze(0).to(d), temps, return_features=True)
        else:
            e_ir = torch.zeros(1, 544, device=d)
        use_gas = gas is not None
        if use_gas:
            g = [[gas["adc"], gas["rs"], gas["rs_r0"], gas["methane_ppm"]]]
            e_gas = self.gas(torch.tensor(self.gas_scaler.transform(g), dtype=torch.float32, device=d), return_features=True)
        else:
            e_gas = torch.zeros(1, 64, device=d)
        probs = F.softmax(self.fusion(e_srgb, e_ir, e_gas, use_ir=use_ir, use_gas=use_gas), dim=1)[0].cpu()
        k = int(probs.argmax())
        label = self.classes[k]
        state = "Spoiled" if label.endswith("_Spoiled") else "Not_spoiled"
        variety = label[: -len("_" + state)]
        return {"label": label, "variety": variety, "state": state, "confidence": float(probs[k]),
                "sensors_used": {"srgb": True, "ir": use_ir, "gas": use_gas}}

# quick self-test on one test sample, with and without sensors
pred = SpoilCheckPredictor(SAVE_DIR, device=str(device))
r = test_df.iloc[0]
print("True label:", r["label_name"])
print("All sensors :", pred.predict(Image.open(r["srgb_path"]), Image.open(r["ir_path"]), r["t_min"], r["t_max"],
                                    {k: r[k] for k in gas_cols}))
print("No IR       :", pred.predict(Image.open(r["srgb_path"]), gas={k: r[k] for k in gas_cols}))
print("Camera only :", pred.predict(Image.open(r["srgb_path"])))
print("\nSaved to", SAVE_DIR, ":", sorted(os.listdir(SAVE_DIR)))